通过使用深度学习框架来简洁地实现线性回归模型，生成数据集

In [1]:
import numpy as np
import torch
from torch.utils import data

def synthetic_data(w,b,num_examples):
    X=torch.normal(0,1,(num_examples,len(w)))#随机生成一组均值为0，标准差为1的x，其中行数为num_examples，列数为len(w)
    y=torch.matmul(X,w)+b#矩阵乘法
    y+=torch.normal(0,0.01,y.shape)#按照y的形状添加N(0,0.01)的噪声
    return X,y.reshape(-1,1)#-1的意思是让pytorch自动计算，因为后面固定为1列

In [2]:
true_w=torch.tensor([2,-3.4])
true_b=4.2
features,labels=synthetic_data(true_w,true_b,1000)

调用框架中现有的api来读取数据

In [3]:
def load_array(data_arrays,batch_size,is_train=True):
    """构造一个pytorch数据迭代器"""
    dataset=data.TensorDataset(*data_arrays)#*把()去掉了，#TensorDataset把两个数据类联系在一起
    return data.DataLoader(dataset,batch_size,shuffle=is_train)#DataLoader可以从dataset中一批一批地抓取数据
batch_size=10
data_iter=load_array((features,labels),batch_size)
next(iter(data_iter))#iter相当于记录可迭代对象的迭代情况，next就是下一次
    

[tensor([[ 2.0548,  0.4447],
         [-0.5804,  1.0661],
         [-0.8995, -0.5207],
         [ 0.1261, -0.9271],
         [ 1.2958,  0.1326],
         [ 2.3520,  0.4456],
         [ 0.8652,  0.2096],
         [-0.2600,  0.0260],
         [ 0.3287, -1.4617],
         [ 2.1946,  0.1996]]),
 tensor([[ 6.7890],
         [-0.5828],
         [ 4.1616],
         [ 7.6015],
         [ 6.3277],
         [ 7.3693],
         [ 5.2065],
         [ 3.5924],
         [ 9.8364],
         [ 7.9159]])]

使用框架的预定好的层

In [7]:
#'nn'是神经网络的缩写
from torch import nn
net=nn.Sequential(nn.Linear(2,1))
#nn.Linear表示构造一个输入为两个特征值，输出为一个特征值的线性模型
#nn.Sequential表示可以将多个神经网络串起来

初始化模型参数

In [8]:
"""权重w随机初始化"""
net[0].weight.data.normal_(0,0.01)#net[0]就是Linear这个神经网络
"""偏差b初始为0"""
net[0].bias.data.fill_(0)

tensor([0.])

计算均方误差使用的是MSELoss类，也称为平方范数

In [9]:
loss=nn.MSELoss()

实例化SGD实例

In [11]:
trainer=torch.optim.SGD(net.parameters(),lr=0.03)#调用优化算法库中的SGD，标明优化的参数和学习率

训练过程代码与我们从零开始实现时所做的非常相似

In [13]:
num_epochs=3
for epoch in range(num_epochs):
    for X,y in data_iter:
        l=loss(net(X),y)#net(X)实际上就是y_hat
        trainer.zero_grad()
        l.backward()
        trainer.step()#表示进行一次trainer
    l=loss(net(features),labels)
    print(f'epoch{epoch+1},loss{l:f}')

epoch1,loss0.000282
epoch2,loss0.000097
epoch3,loss0.000096
